# SFT Results — Global Comparison with Target-Only

3-task scope: **alpaca→samsum**, **tulu3→tydiqa**, **less→mmlu (professional_law)**.

Methods: FullTraining / LayerWiseSubset / GlobalSubset, each at Full / LoRA / MeSO finetuning. 5 seeds {2,22,42,62,82} at swept LRs. Target-only is the in-distribution 16-sample ablation, also at swept LR per (task, FT).

In [1]:
import json
import os
import re
import yaml
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from pathlib import Path
from glob import glob

# Configure matplotlib
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
# ============================================
# Configuration (3-task scope: samsum / tydiqa / less→mmlu prof_law)
# ============================================
SCRATCH_DIR = "/work/hdd/bfwm/phu1/Project"
out_dir = f"{SCRATCH_DIR}/Dr.Post-Training/SFT"
config_base = './train/configs'

model = 'Llama-3.2-1B'

DATASETS = [
    {'train': 'alpaca',  'task': 'samsum', 'subject': '',                 'percentage': 0.4,   'config': 'alpaca_samsum'},
    {'train': 'tulu3',   'task': 'tydiqa', 'subject': '',                 'percentage': 0.01,  'config': 'tulu3_tydiqa'},
    {'train': 'less',    'task': 'mmlu',   'subject': 'professional_law', 'percentage': 0.005, 'config': 'less_mmlu', 'y_clip_max': 8.0},
]

SHARED_CONFIG = {'batch_size': 8, 'n_val': 16}

ALL_METHODS = [
    'FullTraining-Full', 'FullTraining-LoRA', 'FullTraining-MeSO',
    'LayerWiseSubset-Full', 'LayerWiseSubset-LoRA', 'LayerWiseSubset-MeSO',
    'GlobalSubset-Full', 'GlobalSubset-LoRA', 'GlobalSubset-MeSO',
]

FINETUNING_GROUPS = {
    'Full': ['FullTraining-Full', 'GlobalSubset-Full', 'LayerWiseSubset-Full'],
    'LoRA': ['FullTraining-LoRA', 'GlobalSubset-LoRA', 'LayerWiseSubset-LoRA'],
    'MeSO': ['FullTraining-MeSO', 'GlobalSubset-MeSO', 'LayerWiseSubset-MeSO'],
}

FT_ORDER = ['Full', 'LoRA', 'MeSO']
DATASET_TITLES = {
    'alpaca_samsum':              'SAMSUM',
    'tulu3_tydiqa':               'TYDIQA',
    'less_mmlu_professional_law': 'MMLU (prof_law)',
}

LEGEND_NAMES = {
    'FullTraining-Full': 'Full-Training', 'FullTraining-LoRA': 'Full-Training', 'FullTraining-MeSO': 'Full-Training',
    'LayerWiseSubset-Full': 'Layer-Wise Subset', 'LayerWiseSubset-LoRA': 'Layer-Wise Subset', 'LayerWiseSubset-MeSO': 'Layer-Wise Subset',
    'GlobalSubset-Full': 'Global Subset', 'GlobalSubset-LoRA': 'Global Subset', 'GlobalSubset-MeSO': 'Global Subset',
}

METHOD_COLORS = {
    'FullTraining-Full': 'black', 'FullTraining-LoRA': 'black', 'FullTraining-MeSO': 'black',
    'LayerWiseSubset-Full': 'red', 'LayerWiseSubset-LoRA': 'red', 'LayerWiseSubset-MeSO': 'red',
    'GlobalSubset-Full': 'green', 'GlobalSubset-LoRA': 'green', 'GlobalSubset-MeSO': 'green',
}

# For mmlu, the eval task's main run reports per_subject_accuracy (~100 prof_law examples
# inside the full 5062-example test); target-only with --subject filter reports `accuracy`
# directly on prof_law. We extract prof_law accuracy from both for apples-to-apples.
TASK_METRICS = {
    'samsum': {
        'result_file': 'samsum_results.json',
        'metrics': ['rouge1', 'rouge2', 'rougeL'],
        'display_names': ['ROUGE-1', 'ROUGE-2', 'ROUGE-L'],
        'primary_metric': 'rougeL',
        'format': lambda x: f"{x:.3f}" if x is not None else 'N/A',
    },
    'tydiqa': {
        'result_file': 'tydiqa_results.json',
        'metrics': ['f1_score', 'exact_match'],
        'display_names': ['F1 Score', 'Exact Match'],
        'primary_metric': 'f1_score',
        'format': lambda x: f"{x:.2f}" if x is not None else 'N/A',
    },
    'mmlu': {
        'result_file': 'mmlu_results.json',
        'metrics': ['prof_law_accuracy', 'macro_accuracy'],
        'display_names': ['Acc (prof_law)', 'Macro Acc (full)'],
        'primary_metric': 'prof_law_accuracy',
        'format': lambda x: f"{x:.4f}" if x is not None else 'N/A',
    },
}

# Target-only configs (per task × FT, swept-LR ckpts at bs=8)
VAL_ABLATION = [
    {'task': 'samsum', 'subject': '',                 'main_ds_key': 'alpaca_samsum',              'config': 'alpaca_samsum', 'max_steps': 2600, 'batch_size': 8, 'n_val': 16},
    {'task': 'tydiqa', 'subject': '',                 'main_ds_key': 'tulu3_tydiqa',               'config': 'tulu3_tydiqa',  'max_steps': 1174, 'batch_size': 8, 'n_val': 16},
    {'task': 'mmlu',   'subject': 'professional_law', 'main_ds_key': 'less_mmlu_professional_law', 'config': 'less_mmlu',     'max_steps': 1225, 'batch_size': 8, 'n_val': 16, 'y_clip_max': 8.0},
]
VAL_ABLATION_METHODS = ['FullTraining-Full', 'FullTraining-LoRA', 'FullTraining-MeSO']

def _ds_key(ds):
    """Stable key combining train, task, optional subject."""
    k = f"{ds['train']}_{ds['task']}"
    if ds.get('subject'): k = f"{k}_{ds['subject']}"
    return k

def load_lr_from_yaml(config_name, method_name):
    method_yaml = os.path.join(config_base, config_name, f"{method_name}.yaml")
    if os.path.exists(method_yaml):
        with open(method_yaml, 'r') as f:
            cfg = yaml.safe_load(f)
        if cfg and 'learning_rate' in cfg:
            return cfg['learning_rate']
    defaults_yaml = os.path.join(config_base, config_name, 'defaults.yaml')
    if os.path.exists(defaults_yaml):
        with open(defaults_yaml, 'r') as f:
            cfg = yaml.safe_load(f)
        if cfg and 'learning_rate' in cfg:
            return cfg['learning_rate']
    return None

def load_target_only_lr(config_name, ft, subject=''):
    """Read swept target-only LR from Target-only[.subject][.ft].lr.txt"""
    parts = ['Target-only']
    if subject: parts.append(subject)
    if ft != 'Full': parts.append(ft)
    parts.append('lr.txt')
    fname = '.'.join(parts)
    p = os.path.join(config_base, config_name, fname)
    if os.path.exists(p):
        with open(p) as f:
            return f.read().strip()
    return None

# Build LR lookup keyed by ds_key
lr_config = {}
for ds in DATASETS:
    k = _ds_key(ds)
    lr_config[k] = {}
    for method in ALL_METHODS:
        lr = load_lr_from_yaml(ds['config'], method)
        if lr is not None:
            lr_config[k][method] = lr

print(f"Datasets: {[(d['train'], d['task'], d.get('subject') or '-') for d in DATASETS]}\n")
print("Main-run swept LRs:")
for k, methods in lr_config.items():
    print(f"  {k}:")
    for m, lr in methods.items():
        print(f"    {m}: {lr}")
print("\nTarget-only swept LRs:")
for va in VAL_ABLATION:
    for ft in FT_ORDER:
        lr = load_target_only_lr(va['config'], ft, va.get('subject', ''))
        print(f"  {va['main_ds_key']} FullTraining-{ft}: {lr}")

## Helper Functions & Load All Data

In [ ]:
# ============================================
# Helper functions + load main-run data
# ============================================

def _finetuning_type(name):
    return name.split('-', 1)[1]

def _format_lr(lr_val):
    """Format LR value to match directory naming convention."""
    if lr_val is None: return '5e-05'
    lr_val = float(lr_val)
    return f'{lr_val:.2e}' if lr_val < 0.001 else f'{lr_val}'

def _lookup_lr(method_name, ds):
    k = _ds_key(ds)
    if k in lr_config and method_name in lr_config[k]:
        return _format_lr(lr_config[k][method_name])
    ft = _finetuning_type(method_name)
    return '2e-04' if ft == 'LoRA' else '5e-05'

def _dir_name(method_name, seed, ds):
    """Main-run dir name. Subject-AGNOSTIC even for less→mmlu (model is trained on
    LESS-mix without subject filter; subject only affects eval)."""
    lr = _lookup_lr(method_name, ds)
    cfg = SHARED_CONFIG
    return (f"{ds['train']}_{ds['task']}-{model}-{method_name}"
            f"-p{ds['percentage']}-lr{lr}-b{cfg['batch_size']}-v{cfg['n_val']}-s{seed}")

def _discover_seeds(method_name, ds):
    pattern = _dir_name(method_name, '*', ds)
    seeds = []
    for p in glob(f"{out_dir}/{pattern}"):
        m = re.search(r'-s(\d+)$', Path(p).name)
        if m: seeds.append(int(m.group(1)))
    return sorted(seeds)

def _load_eval_results(path):
    with open(path) as f: data = json.load(f)
    steps = [item['step'] for item in data]
    val_ppl = [item['val_perplexity'] for item in data]
    eval_ppl = [item['eval_perplexity'] for item in data]
    wt = [item.get('wall_time') for item in data]
    twt = [item.get('train_wall_time', item.get('wall_time')) for item in data]
    return {
        'steps': np.array(steps),
        'val_perplexity': np.array(val_ppl),
        'eval_perplexity': np.array(eval_ppl),
        'wall_time': np.array(wt) if all(t is not None for t in wt) else None,
        'train_wall_time': np.array(twt) if all(t is not None for t in twt) else None,
    }

def load_method_results(method_name, ds):
    seeds = _discover_seeds(method_name, ds)
    if not seeds: return None
    all_res, valid_seeds = [], []
    for seed in seeds:
        path = f"{out_dir}/{_dir_name(method_name, seed, ds)}/evaluation_results.json"
        if Path(path).exists():
            all_res.append(_load_eval_results(path))
            valid_seeds.append(seed)
    if not all_res: return None
    n = len(valid_seeds); se = np.sqrt(n)
    min_len = min(len(r['steps']) for r in all_res)
    out = {'steps': all_res[0]['steps'][:min_len], 'n_seeds': n, 'seeds': valid_seeds}
    for k in ['val_perplexity', 'eval_perplexity']:
        st = np.stack([r[k][:min_len] for r in all_res])
        out[f'{k}_mean'] = np.mean(st, 0); out[f'{k}_std'] = np.std(st, 0) / se
    for k in ['wall_time', 'train_wall_time']:
        vals = [r[k] for r in all_res if r[k] is not None]
        if len(vals) == n:
            st = np.stack([w[:min_len] for w in vals])
            out[f'{k}_mean'] = np.mean(st, 0); out[f'{k}_std'] = np.std(st, 0) / se
        else:
            out[f'{k}_mean'] = None; out[f'{k}_std'] = None
    return out

def _extract_metric(result_json, task, metric, subject=''):
    """Pull a specific metric. For mmlu prof_law_accuracy, look in per_subject_accuracy."""
    if task == 'mmlu' and metric == 'prof_law_accuracy':
        # main runs: prof_law accuracy lives in per_subject_accuracy['professional_law']
        # target-only runs: were eval'd with --subject=prof_law, so 'accuracy' IS prof_law
        psa = result_json.get('per_subject_accuracy') or {}
        if 'professional_law' in psa:
            return psa['professional_law']
        return result_json.get('accuracy')
    return result_json.get(metric)

def load_task_metric(method_name, ds, seeds):
    task = ds['task']
    if task not in TASK_METRICS: return None
    rfile = TASK_METRICS[task]['result_file']
    seed_results, valid = [], []
    for seed in seeds:
        path = f"{out_dir}/{_dir_name(method_name, seed, ds)}/{rfile}"
        if Path(path).exists():
            with open(path) as f: seed_results.append(json.load(f)); valid.append(seed)
    if not seed_results: return None
    agg = {'seeds': valid, 'n_seeds': len(valid)}
    for metric in TASK_METRICS[task]['metrics']:
        vals = [_extract_metric(r, task, metric, ds.get('subject', '')) for r in seed_results]
        vals = [v for v in vals if v is not None]
        if vals:
            agg[f'{metric}_mean'] = np.mean(vals); agg[f'{metric}_std'] = np.std(vals) / np.sqrt(len(vals))
        else:
            agg[f'{metric}_mean'] = None; agg[f'{metric}_std'] = None
    return agg

# ============================================
# Load all main-run data
# ============================================
all_results = {}
all_task_results = {}
for ds in DATASETS:
    k = _ds_key(ds)
    print(f"\n{'='*60}\nLoading: {k}\n{'='*60}")
    res, tres = {}, {}
    for m in ALL_METHODS:
        seeds = _discover_seeds(m, ds)
        d = load_method_results(m, ds)
        if d:
            res[m] = d
            print(f"  + {m}: {d['n_seeds']} seeds, {len(d['steps'])} pts", end='')
        elif seeds:
            print(f"  ~ {m}: {len(seeds)} seeds (no curves)", end='')
        else:
            print(f"  - {m}: not found"); continue
        td = load_task_metric(m, ds, seeds)
        if td: tres[m] = td; print(f"  | task: {td['n_seeds']} seeds")
        else: print()
    all_results[k] = res; all_task_results[k] = tres
    print(f"  Loaded {len(res)}/{len(ALL_METHODS)} training, {len(tres)} task-specific")

In [ ]:
# ============================================
# Target-only: dir naming + LR lookup (handles subject + per-FT swept LRs)
# ============================================

def _va_dir_name(method_name, seed, va):
    """{task}{_subj}_val_{task}{_subj}-{model}-{method}-ms{ms}-lr{lr}-b{bs}-v{nv}-s{seed}"""
    ft = _finetuning_type(method_name)
    raw_lr = load_target_only_lr(va['config'], ft, va.get('subject', ''))
    if raw_lr is None: return None  # signals missing LR file
    lr = _format_lr(raw_lr)
    bs = va.get('batch_size', 8); nv = va.get('n_val', 16)
    subj_tag = f"_{va['subject']}" if va.get('subject') else ''
    return (f"{va['task']}{subj_tag}_val_{va['task']}{subj_tag}-{model}-{method_name}"
            f"-ms{va['max_steps']}-lr{lr}-b{bs}-v{nv}-s{seed}")

def _va_discover_seeds(method_name, va):
    pat = _va_dir_name(method_name, '*', va)
    if pat is None: return []
    seeds = []
    for p in glob(f"{out_dir}/{pat}"):
        m = re.search(r'-s(\d+)$', Path(p).name)
        if m: seeds.append(int(m.group(1)))
    return sorted(seeds)

def load_va_results(method_name, va):
    seeds = _va_discover_seeds(method_name, va)
    if not seeds: return None
    runs, valid = [], []
    for s in seeds:
        path = f"{out_dir}/{_va_dir_name(method_name, s, va)}/evaluation_results.json"
        if Path(path).exists():
            runs.append(_load_eval_results(path)); valid.append(s)
    if not runs: return None
    n = len(valid); se = np.sqrt(n)
    min_len = min(len(r['steps']) for r in runs)
    out = {'steps': runs[0]['steps'][:min_len], 'n_seeds': n, 'seeds': valid}
    for k in ['val_perplexity', 'eval_perplexity']:
        st = np.stack([r[k][:min_len] for r in runs])
        out[f'{k}_mean'] = np.mean(st, 0); out[f'{k}_std'] = np.std(st, 0) / se
    for k in ['wall_time', 'train_wall_time']:
        vals = [r[k] for r in runs if r[k] is not None]
        if len(vals) == n:
            st = np.stack([w[:min_len] for w in vals])
            out[f'{k}_mean'] = np.mean(st, 0); out[f'{k}_std'] = np.std(st, 0) / se
        else:
            out[f'{k}_mean'] = None; out[f'{k}_std'] = None
    return out

def load_va_task_metric(method_name, va, seeds):
    task = va['task']
    if task not in TASK_METRICS: return None
    rfile = TASK_METRICS[task]['result_file']
    seed_results, valid = [], []
    for s in seeds:
        path = f"{out_dir}/{_va_dir_name(method_name, s, va)}/{rfile}"
        if Path(path).exists():
            with open(path) as f: seed_results.append(json.load(f)); valid.append(s)
    if not seed_results: return None
    agg = {'seeds': valid, 'n_seeds': len(valid)}
    for metric in TASK_METRICS[task]['metrics']:
        vals = [_extract_metric(r, task, metric, va.get('subject', '')) for r in seed_results]
        vals = [v for v in vals if v is not None]
        if vals:
            agg[f'{metric}_mean'] = np.mean(vals); agg[f'{metric}_std'] = np.std(vals) / np.sqrt(len(vals))
        else:
            agg[f'{metric}_mean'] = None; agg[f'{metric}_std'] = None
    return agg

val_ablation_results = {}
val_ablation_task_results = {}
for va in VAL_ABLATION:
    k = va['main_ds_key']
    print(f"\n{'='*60}\nLoading target-only for {k}\n{'='*60}")
    res, tres = {}, {}
    for m in VAL_ABLATION_METHODS:
        d = load_va_results(m, va)
        if d:
            res[m] = d
            seeds = d['seeds']
            td = load_va_task_metric(m, va, seeds)
            if td:
                tres[m] = td
                print(f"  + {m}: {d['n_seeds']} seeds | metrics: {[k for k in td if '_mean' in k]}")
            else:
                print(f"  + {m}: {d['n_seeds']} seeds (no task metric)")
        else:
            print(f"  - {m}: not found")
    val_ablation_results[k] = res
    val_ablation_task_results[k] = tres
    print(f"  Loaded {len(res)}/{len(VAL_ABLATION_METHODS)} training, {len(tres)} task-specific")

In [ ]:
# ============================================
# Global Comparison Grid (vs Step) — 3 cols × 3 rows (FT types)
# ============================================
ncols = len(DATASETS); nrows = len(FT_ORDER)
fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 4.5*nrows), squeeze=False)
fig.patch.set_facecolor('white')

for col_idx, ds in enumerate(DATASETS):
    k = _ds_key(ds)
    results = all_results[k]
    va_results = val_ablation_results.get(k, {})
    for row_idx, ft_name in enumerate(FT_ORDER):
        ax = axes[row_idx, col_idx]; ax.set_facecolor('white')
        for m in FINETUNING_GROUPS[ft_name]:
            if m not in results: continue
            d = results[m]; mean = d['eval_perplexity_mean']; std = d['eval_perplexity_std']
            ax.plot(d['steps'], mean, linewidth=2, label=LEGEND_NAMES[m], color=METHOD_COLORS[m])
            if d['n_seeds'] > 1:
                ax.fill_between(d['steps'], mean-std, mean+std, color=METHOD_COLORS[m], alpha=0.2)
        # Target-only overlay for this FT row
        va_method = f"FullTraining-{ft_name}"
        if va_method in va_results:
            d = va_results[va_method]
            mean = d['eval_perplexity_mean']; std = d['eval_perplexity_std']
            ax.plot(d['steps'], mean, linewidth=2, label='Target-only', color='blue', linestyle='--')
            if d['n_seeds'] > 1:
                ax.fill_between(d['steps'], mean-std, mean+std, color='blue', alpha=0.15)
        if row_idx == 0:
            ax.set_title(DATASET_TITLES.get(k, k), fontsize=22, fontweight='bold', pad=10)
        if row_idx == nrows-1: ax.set_xlabel('Step', fontsize=18)
        if col_idx == 0: ax.set_ylabel(f'{ft_name}\nEval Perplexity', fontsize=18)
        ax.grid(True, alpha=0.3, linewidth=0.5)
        for s in ax.spines.values(): s.set_visible(True); s.set_linewidth(1.0); s.set_color('black')
        ax.tick_params(axis='both', which='major', labelsize=14, direction='out', length=4)
        ax.xaxis.set_major_locator(plt.MaxNLocator(5)); ax.yaxis.set_major_locator(plt.MaxNLocator(5))

# Shared y limits per column with y_clip cap
for col_idx in range(ncols):
    col_axes = [axes[r, col_idx] for r in range(nrows)]
    sx = (min(a.get_xlim()[0] for a in col_axes), max(a.get_xlim()[1] for a in col_axes))
    sy = (min(a.get_ylim()[0] for a in col_axes), max(a.get_ylim()[1] for a in col_axes))
    y_cap = DATASETS[col_idx].get('y_clip_max')
    if y_cap is not None: sy = (sy[0], min(sy[1], y_cap))
    for a in col_axes: a.set_xlim(sx); a.set_ylim(sy)

handles, labels = [], []
for r in axes:
    for a in r:
        h, l = a.get_legend_handles_labels(); handles += h; labels += l
seen = {}; uh, ul = [], []
for h, l in zip(handles, labels):
    if l not in seen: seen[l] = True; uh.append(h); ul.append(l)
fig.legend(uh, ul, loc='lower center', ncol=len(ul), fontsize=18,
           frameon=True, edgecolor='black', fancybox=False, framealpha=1.0,
           bbox_to_anchor=(0.5, -0.02))
plt.tight_layout(rect=[0, 0.05, 1, 1]); plt.show()

In [ ]:
# ============================================
# Global Comparison Grid (vs Wall Time)
# ============================================
ncols = len(DATASETS); nrows = len(FT_ORDER)
fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 4.5*nrows), squeeze=False)
fig.patch.set_facecolor('white')

for col_idx, ds in enumerate(DATASETS):
    k = _ds_key(ds)
    results = all_results[k]
    va_results = val_ablation_results.get(k, {})
    for row_idx, ft_name in enumerate(FT_ORDER):
        ax = axes[row_idx, col_idx]; ax.set_facecolor('white')
        for m in FINETUNING_GROUPS[ft_name]:
            if m not in results: continue
            d = results[m]
            wt = d['train_wall_time_mean'] if d['train_wall_time_mean'] is not None else d['wall_time_mean']
            if wt is None: continue
            x = wt / 60.0; mean = d['eval_perplexity_mean']; std = d['eval_perplexity_std']
            ax.plot(x, mean, linewidth=2, label=LEGEND_NAMES[m], color=METHOD_COLORS[m])
            if d['n_seeds'] > 1:
                ax.fill_between(x, mean-std, mean+std, color=METHOD_COLORS[m], alpha=0.2)
        va_method = f"FullTraining-{ft_name}"
        if va_method in va_results:
            d = va_results[va_method]
            wt = d['train_wall_time_mean'] if d['train_wall_time_mean'] is not None else d['wall_time_mean']
            if wt is not None:
                x = wt / 60.0; mean = d['eval_perplexity_mean']; std = d['eval_perplexity_std']
                ax.plot(x, mean, linewidth=2, label='Target-only', color='blue', linestyle='--')
                if d['n_seeds'] > 1:
                    ax.fill_between(x, mean-std, mean+std, color='blue', alpha=0.15)
        if row_idx == 0:
            ax.set_title(DATASET_TITLES.get(k, k), fontsize=22, fontweight='bold', pad=10)
        if row_idx == nrows-1: ax.set_xlabel('Wall Time (minutes)', fontsize=18)
        if col_idx == 0: ax.set_ylabel(f'{ft_name}\nEval Perplexity', fontsize=18)
        ax.grid(True, alpha=0.3, linewidth=0.5)
        for s in ax.spines.values(): s.set_visible(True); s.set_linewidth(1.0); s.set_color('black')
        ax.tick_params(axis='both', which='major', labelsize=14, direction='out', length=4)
        ax.xaxis.set_major_locator(plt.MaxNLocator(5)); ax.yaxis.set_major_locator(plt.MaxNLocator(5))

for col_idx in range(ncols):
    col_axes = [axes[r, col_idx] for r in range(nrows)]
    sx = (min(a.get_xlim()[0] for a in col_axes), max(a.get_xlim()[1] for a in col_axes))
    sy = (min(a.get_ylim()[0] for a in col_axes), max(a.get_ylim()[1] for a in col_axes))
    y_cap = DATASETS[col_idx].get('y_clip_max')
    if y_cap is not None: sy = (sy[0], min(sy[1], y_cap))
    for a in col_axes: a.set_xlim(sx); a.set_ylim(sy)

handles, labels = [], []
for r in axes:
    for a in r:
        h, l = a.get_legend_handles_labels(); handles += h; labels += l
seen = {}; uh, ul = [], []
for h, l in zip(handles, labels):
    if l not in seen: seen[l] = True; uh.append(h); ul.append(l)
fig.legend(uh, ul, loc='lower center', ncol=len(ul), fontsize=18,
           frameon=True, edgecolor='black', fancybox=False, framealpha=1.0,
           bbox_to_anchor=(0.5, -0.02))
plt.tight_layout(rect=[0, 0.05, 1, 1]); plt.show()

In [ ]:
# ============================================
# Global Table: All Methods + Target-Only (consolidated, 3 tasks)
# ============================================
def _fmt(mean, std, n_seeds, fmt_fn=None):
    if mean is None: return 'N/A'
    base = fmt_fn(mean) if fmt_fn else f"{mean:.4f}"
    if n_seeds > 1 and std is not None:
        return f"{mean:.4f} +/- {std:.4f}" if fmt_fn is None else f"{mean:.3f} +/- {std:.3f}"
    return base

VA_DISPLAY = {
    'FullTraining-Full': 'Target-only (Full)',
    'FullTraining-LoRA': 'Target-only (LoRA)',
    'FullTraining-MeSO': 'Target-only (MeSO)',
}

for ds in DATASETS:
    k = _ds_key(ds); task = ds['task']
    main_t = all_task_results.get(k, {}); va_t = val_ablation_task_results.get(k, {})
    if task not in TASK_METRICS: continue
    tm = TASK_METRICS[task]; primary = tm['primary_metric']

    print(f"\n{'='*100}")
    print(f"  {DATASET_TITLES.get(k, k)} ({task.upper()}{('/' + ds['subject']) if ds.get('subject') else ''})")
    print(f"{'='*100}")
    header = f"{'Method':<25} {'Seeds':<7}"
    for dn in tm['display_names']: header += f" {dn:>20}"
    print(header); print("-" * 100)

    for name, d in sorted(main_t.items(),
                           key=lambda x: x[1].get(f'{primary}_mean', 0) or 0, reverse=True):
        n = d['n_seeds']; row = f"{name:<25} {n:<7}"
        for metric in tm['metrics']:
            row += f" {_fmt(d.get(f'{metric}_mean'), d.get(f'{metric}_std'), n, tm['format']):>20}"
        print(row)

    if va_t:
        print(f"{'- target-only ':-<100}")
        for name, d in sorted(va_t.items(),
                               key=lambda x: x[1].get(f'{primary}_mean', 0) or 0, reverse=True):
            n = d['n_seeds']; display = VA_DISPLAY.get(name, name); row = f"{display:<25} {n:<7}"
            for metric in tm['metrics']:
                row += f" {_fmt(d.get(f'{metric}_mean'), d.get(f'{metric}_std'), n, tm['format']):>20}"
            print(row)
    print(f"{'='*100}")